# Классическое Компьютерное Зрение (Classical CV)

В этом ноутбуке мы разберём **классические методы** компьютерного зрения — те, что использовались до эпохи глубокого обучения.

**Что ты изучишь:**
1. Обнаружение ключевых точек (SIFT, ORB, SURF)
2. Описание признаков и matching
3. RANSAC и robust estimation
4. Построение панорам
5. Сравнение с глубокими методами

> **Запускай в Colab** — Runtime → T4 GPU.

## Почему важно знать классические методы?

Даже в 2026 году классические алгоритмы всё ещё используются:
- Когда данных мало
- Когда нужна **интерпретируемость**
- В комбинации с глубокими методами (hybrid approaches)
- На embedded устройствах (быстрее и легче)

# 1. Установка OpenCV

In [ ]:
!pip install --quiet opencv-python opencv-contrib-python

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import requests
from io import BytesIO

# 2. Обнаружение ключевых точек

In [ ]:
# Загружаем изображение
url = 'https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/1200px-Cat03.jpg'
image = np.array(Image.open(BytesIO(requests.get(url).content)))
gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)

# SIFT
sift = cv2.SIFT_create()
keypoints_sift, descriptors_sift = sift.detectAndCompute(gray, None)

# ORB (быстрее и бесплатный)
orb = cv2.ORB_create()
keypoints_orb, descriptors_orb = orb.detectAndCompute(gray, None)

print(f"SIFT точек: {len(keypoints_sift)}")
print(f"ORB точек:  {len(keypoints_orb)}")

# 3. Matching ключевых точек

In [ ]:
# Пример matching между двумя изображениями
img1 = cv2.imread('image1.jpg', 0)
img2 = cv2.imread('image2.jpg', 0)

bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
matches = bf.match(descriptors_orb, descriptors_orb)
matches = sorted(matches, key=lambda x: x.distance)

# Рисуем лучшие 20 совпадений
result = cv2.drawMatches(img1, keypoints_orb, img2, keypoints_orb, matches[:20], None, flags=2)
plt.imshow(result)
plt.show()

# 4. RANSAC и Homography

In [ ]:
# Нахождение гомографии между двумя изображениями
src_pts = np.float32([keypoints_orb[m.queryIdx].pt for m in matches]).reshape(-1, 1, 2)
dst_pts = np.float32([keypoints_orb[m.trainIdx].pt for m in matches]).reshape(-1, 1, 2)

M, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 5.0)

print("Гомография найдена!")

# 5. Построение панорамы

In [ ]:
# Простая панорама из двух изображений
height, width = img1.shape
warped = cv2.warpPerspective(img2, M, (width * 2, height))

panorama = np.zeros((height, width * 2), dtype=np.uint8)
panorama[0:height, 0:width] = img1
panorama[0:height, width:width*2] = warped[0:height, width:width*2]

plt.imshow(panorama, cmap='gray')
plt.title('Панорама')
plt.show()

# 6. Сравнение с глубокими методами

| Метод       | Скорость | Точность | Интерпретируемость | Нужна разметка |
|-------------|----------|----------|--------------------|----------------|
| SIFT/ORB    | ★★★★★   | ★★★      | ★★★★★             | Нет            |
| CNN (YOLO)  | ★★★★    | ★★★★★    | ★★                 | Да             |
| Foundation  | ★★★     | ★★★★★    | ★                  | Нет (zero-shot)|

**Вывод:** Классические методы до сих пор отличный выбор, когда данных мало или нужна скорость/интерпретируемость.

# 7. Упражнения

### Упражнение 1: Сделай matching между своими фото

Сфотографируй один объект с двух разных ракурсов и найди совпадающие точки.

In [ ]:
# Твой код здесь

### Упражнение 2: Построй панораму из 3+ изображений

In [ ]:
# Твой код здесь

### Упражнение 3: Сравни SIFT vs ORB

Сравни количество точек, время работы и качество matching.

---

**Готово!**  
Ты освоил классические методы компьютерного зрения — основу, на которой построено всё современное CV.

Поздравляю! Ты прошёл полный курс **Deep Learning and Computer Vision** от основ до foundation models.